## Tutorial 9, Question 2

In this tutorial, we will:

1. Construct a causal attention mask.
2. Demonstrate why a Transformer decoder needs causal masking.
3. Verify that masked attention weights are zero.
4. Examine the tensor shapes produced by multi-head attention.

A Transformer decoder generates a sequence one token at a time. When predicting the token at position $t$, it must not use information from positions after $t$.

For example, when predicting the third token, the model may attend to the first, second, and third positions, but not to the fourth position.

In [1]:
import torch
import torch.nn as nn

torch.set_printoptions(
    precision=4,
    sci_mode=False
)

# Make the experiment reproducible
torch.manual_seed(2)

## 1. Create the input sequence

We create an input tensor with shape

$$
[\text{batch size},\ \text{sequence length},\ \text{embedding dimension}]
=
[1,4,8].
$$

This represents:

- one input sequence;
- four token positions;
- an eight-dimensional embedding for each token.

We then make a copy of the input and change only the final token.

The operation

```python
X_changed[:, -1] += 10.0
```

adds 10 to every embedding dimension of the last token. The first three tokens remain exactly the same.

This creates a controlled experiment:

- `X` is the original sequence;
- `X_changed` differs only at the final position.

We will test whether changing the final token affects the outputs at earlier positions.

In [2]:
# Shape: [batch size, sequence length, embedding dimension]
X = torch.randn(1, 4, 8)

# Make an independent copy of X
X_changed = X.clone()

# Change only the final token
X_changed[:, -1] += 10.0

print("Input shape:", X.shape)

print("\nOriginal final token:")
print(X[:, -1])

print("\nChanged final token:")
print(X_changed[:, -1])

print(
    "\nAre positions 0-2 unchanged?",
    torch.equal(X[:, :-1], X_changed[:, :-1])
)

Input shape: torch.Size([1, 4, 8])

Original final token:
tensor([[ 0.4640, -0.4986,  0.1289,  2.7631,  0.1405,  1.1191,  0.3152,  1.7528]])

Changed final token:
tensor([[10.4640,  9.5014, 10.1289, 12.7631, 10.1405, 11.1191, 10.3152, 11.7528]])

Are positions 0-2 unchanged? True


## 2. Create the multi-head attention layer

We create an attention layer with:

- embedding dimension $D=8$;
- two attention heads;
- four dimensions per head;
- no dropout.

The dimension processed by each head is

$$
D_{\text{head}}
=
\frac{D}{H}
=
\frac{8}{2}
=
4.
$$

Setting `batch_first=True` means that the layer expects tensors with the shape

$$
[\text{batch size},\ \text{sequence length},\ \text{embedding dimension}].
$$

Calling `attention.eval()` places the layer in evaluation mode. Together with `dropout=0.0`, this ensures that the same input always produces the same output.

In [3]:
attention = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=2,
    dropout=0.0,
    batch_first=True
)

attention.eval()

print("Embedding dimension:", attention.embed_dim)
print("Number of heads:", attention.num_heads)
print(
    "Dimension per head:",
    attention.embed_dim // attention.num_heads
)

Embedding dimension: 8
Number of heads: 2
Dimension per head: 4


## 3. Construct a causal mask

For a sequence of four tokens, the causal mask has shape

$$
[4,4].
$$

The rows represent **query positions**, while the columns represent **key positions**.

For a Boolean mask used by `nn.MultiheadAttention`:

- `False` means attention is allowed;
- `True` means attention is blocked.

The desired mask is

| Query position | Key 0 | Key 1 | Key 2 | Key 3 |
|---:|:---:|:---:|:---:|:---:|
| 0 | Allowed | Blocked | Blocked | Blocked |
| 1 | Allowed | Allowed | Blocked | Blocked |
| 2 | Allowed | Allowed | Allowed | Blocked |
| 3 | Allowed | Allowed | Allowed | Allowed |

For example:

- Query 0 can attend only to key 0.
- Query 1 can attend to keys 0 and 1.
- Query 2 can attend to keys 0, 1, and 2.
- Query 3 can attend to all four keys.

`torch.triu(..., diagonal=1)` selects the part strictly above the main diagonal. These are exactly the future positions that must be hidden.

In [4]:
sequence_length = X.shape[1]

# True means that attention is not allowed
causal_mask = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool
    ),
    diagonal=1
)

print("Causal mask:")
print(causal_mask)

print("\nMask shape:", causal_mask.shape)

Causal mask:
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

Mask shape: torch.Size([4, 4])


The mask printed by the previous cell should be

```text
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])
```

Consider row 2:

```text
[False, False, False, True]
```

The token at position 2 may attend to positions 0, 1, and 2. It is prohibited from attending to position 3 because position 3 is in the future.

## 4. Self-attention without causal masking

First, we apply self-attention without a mask.

In self-attention, the same sequence provides the query, key, and value:

$$
Q=XW_Q,\qquad
K=XW_K,\qquad
V=XW_V.
$$

We apply the same attention layer to both `X` and `X_changed`.

Because no mask is used, every query position can attend to every key position. Therefore, positions 0–2 can access the changed final token.

In [5]:
# Self-attention for the original sequence
Y_full, _ = attention(
    query=X,
    key=X,
    value=X,
    need_weights=False
)

# Self-attention after changing the final token
Y_full_changed, _ = attention(
    query=X_changed,
    key=X_changed,
    value=X_changed,
    need_weights=False
)

print("Input shape:", X.shape)
print("Output shape:", Y_full.shape)

Input shape: torch.Size([1, 4, 8])
Output shape: torch.Size([1, 4, 8])


## 5. Measure the effect of changing the final token

We compare the outputs before and after changing the final token.

The expression

```python
Y_full_changed[:, :-1] - Y_full[:, :-1]
```

compares only positions 0–2.

Here:

- `:` selects every item in the batch;
- `:-1` selects all sequence positions except the final one;
- the embedding dimension is left unchanged.

We exclude the final output because the final input itself was deliberately changed. We are interested in whether this change leaks into the earlier positions.

The maximum absolute difference is calculated as

$$
\max\left|
Y_{\text{changed},\,0:2}
-
Y_{\text{original},\,0:2}
\right|.
$$

In [6]:
full_difference = (
    Y_full_changed[:, :-1] - Y_full[:, :-1]
).abs().max()

# Maximum difference for each sequence position
full_difference_per_position = (
    Y_full_changed - Y_full
).abs().amax(dim=-1).squeeze(0)

print(
    "Maximum change at positions 0-2 without mask:",
    full_difference.item()
)

print("\nChange at each position without mask:")
for position, difference in enumerate(
    full_difference_per_position
):
    print(
        f"Position {position}: "
        f"{difference.item():.6f}"
    )

Maximum change at positions 0-2 without mask: 8.518075942993164

Change at each position without mask:
Position 0: 8.430142
Position 1: 8.298594
Position 2: 8.518076
Position 3: 0.186714


### Interpretation

The difference at positions 0–2 should be greater than zero.

This occurs because, without a causal mask, every token can attend to the final token. Changing the final token therefore changes:

1. its key vector;
2. its value vector;
3. the attention scores involving that token;
4. the weighted values received by earlier positions.

The earlier outputs can therefore change even though their own input embeddings were not modified.

## 6. Apply causal masking

We now repeat the experiment using

```python
attn_mask=causal_mask
```

This prohibits each query from attending to future keys.

We also request the attention weights:

```python
need_weights=True
```

Setting

```python
average_attn_weights=False
```

keeps the weights from the two attention heads separate.

The returned attention-weight tensor has shape

$$
[B,H,L,S],
$$

where:

- $B$ is the batch size;
- $H$ is the number of attention heads;
- $L$ is the target or query sequence length;
- $S$ is the source or key sequence length.

In this self-attention example, its shape is

$$
[1,2,4,4].
$$

In [7]:
# Masked self-attention for the original sequence
Y_masked, masked_weights = attention(
    query=X,
    key=X,
    value=X,
    attn_mask=causal_mask,
    need_weights=True,
    average_attn_weights=False
)

# Masked self-attention after changing the final token
Y_masked_changed, _ = attention(
    query=X_changed,
    key=X_changed,
    value=X_changed,
    attn_mask=causal_mask,
    need_weights=True,
    average_attn_weights=False
)

print("Masked output shape:", Y_masked.shape)
print("Attention-weight shape:", masked_weights.shape)

Masked output shape: torch.Size([1, 4, 8])
Attention-weight shape: torch.Size([1, 2, 4, 4])


## 7. Compare the masked outputs

With causal masking:

- position 0 cannot attend to the changed token at position 3;
- position 1 cannot attend to position 3;
- position 2 cannot attend to position 3;
- position 3 can attend to itself and can therefore be affected.

Consequently, changing the final token should not affect the outputs at positions 0–2.

In [8]:
masked_difference = (
    Y_masked_changed[:, :-1] - Y_masked[:, :-1]
).abs().max()

masked_difference_per_position = (
    Y_masked_changed - Y_masked
).abs().amax(dim=-1).squeeze(0)

print(
    "Maximum change at positions 0-2 with mask:",
    masked_difference.item()
)

print("\nChange at each position with mask:")
for position, difference in enumerate(
    masked_difference_per_position
):
    print(
        f"Position {position}: "
        f"{difference.item():.6f}"
    )

Maximum change at positions 0-2 with mask: 0.0

Change at each position with mask:
Position 0: 0.000000
Position 1: 0.000000
Position 2: 0.000000
Position 3: 0.186714


### Expected observation

The expected pattern is:

```text
Change at positions 0-2 without mask: greater than 0
Change at positions 0-2 with mask:    approximately 0
```

The final position can still change because it is allowed to attend to itself.

This experiment demonstrates the main purpose of causal masking:

> A prediction at position $t$ must depend only on positions up to and including $t$.

Mathematically, the output at position $t$ may depend on

$$
x_0,x_1,\ldots,x_t,
$$

but not on

$$
x_{t+1},x_{t+2},\ldots.
$$

## 8. Inspect the masked attention weights

Because the layer contains two heads, the attention weights have shape

$$
[1,2,4,4].
$$

For example,

```python
masked_weights[0, 0]
```

selects:

- batch item 0;
- attention head 0;
- all four query positions;
- all four key positions.

Every row should sum to one because Softmax normalizes the attention weights across the key positions.

All entries above the main diagonal should be zero because they correspond to future tokens.

In [9]:
for head in range(masked_weights.shape[1]):
    print(f"Attention weights for head {head}:")
    print(masked_weights[0, head])

    print(
        "Row sums:",
        masked_weights[0, head].sum(dim=-1)
    )

    print()

Attention weights for head 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4940, 0.5060, 0.0000, 0.0000],
        [0.2447, 0.3209, 0.4344, 0.0000],
        [0.1538, 0.2996, 0.3486, 0.1980]], grad_fn=<SelectBackward0>)
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)

Attention weights for head 1:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3598, 0.6402, 0.0000, 0.0000],
        [0.1857, 0.3164, 0.4979, 0.0000],
        [0.7287, 0.1522, 0.0888, 0.0304]], grad_fn=<SelectBackward0>)
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)



## Causal self-attention

In causal self-attention:

- $Q$, $K$, and $V$ come from the decoder sequence.
- Each position may attend only to itself and earlier positions.
- Changing a future token cannot affect earlier outputs.
- Attention weights assigned to masked future positions are zero.
